# [FD002] Análisis Exploratorio de Datos (EDA) Orientado a Diagnóstico y Pronóstico de RUL

**Dataset:** C-MAPSS Subconjunto FD002 (NASA Ames Prognostics Center of Excellence)  
**Condiciones operativas:** 6 regímenes (combinaciones de Altitud, Mach, TRA)  
**Modo de falla:** Degradación del compresor de alta presión (HPC - High Pressure Compressor)

---

## Objetivo General
Establecer la fundamentación metodológica para el subconjunto **FD002**. A diferencia del FD001, este dataset presenta alta variabilidad operativa debido a la presencia de **6 regímenes** de operación distintos, los cuales enmascaran la tendencia de degradación de los sensores. Este EDA se centra en:

1. **Clusterización de Regímenes:** Identificación automática de los 6 modos operativos mediante K-Means sobre las variables `setting_1`, `setting_2` y `setting_3`.
2. **Análisis de Varianza Intra-Régimen:** Cuantificación de la variabilidad del ruido operativo para distinguir entre perturbación de régimen y degradación real.
3. **Remoción de Sensores Invariables:** Identificación de sensores con varianza constante incluso dentro de los regímenes.
4. **Visualización de Degradación:** Análisis de la relación entre sensores clave y el RUL Piecewise dentro de regímenes específicos.


In [ ]:
# Configuración Inicial e Importación de Módulos
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Configuración estilo publicación (IEEE/Nature)
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10,
    'axes.labelsize': 11, 'axes.titlesize': 12,
    'figure.dpi': 100, 'savefig.dpi': 300,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--'
})
sns.set_palette('colorblind')

# Rutas
PROJECT_ROOT = Path(os.path.abspath('..')) if os.path.exists('../datos') else Path(os.path.abspath('.'))
DATA_DIR = PROJECT_ROOT / 'datos'
FIGURES_DIR = PROJECT_ROOT / 'notebooks' / 'figures' / 'FD002'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def guardar_fig(nombre, fig=None):
    """Guarda la figura en notebooks/figures/FD002 como PNG (300 dpi) y PDF."""
    if fig is None:
        fig = plt.gcf()
    png = FIGURES_DIR / f'{nombre}.png'
    pdf = FIGURES_DIR / f'{nombre}.pdf'
    fig.savefig(png, dpi=300, bbox_inches='tight')
    fig.savefig(pdf, bbox_inches='tight')
    print(f'  Guardado: {png.relative_to(PROJECT_ROOT)} (+ .pdf)')
    return png

# Columnas
INDEX_COLS = ['unit_number', 'time_cycles']
SETTING_COLS = ['setting_1', 'setting_2', 'setting_3']
SENSOR_COLS = [f'sensor_{i}' for i in range(1, 22)]
COLUMN_NAMES = INDEX_COLS + SETTING_COLS + SENSOR_COLS

print('Configuración cargada.')
print(f'Ruta del proyecto: {PROJECT_ROOT}')
print(f'Ruta de los datos: {DATA_DIR}')
print(f'Ruta de figuras:   {FIGURES_DIR}')
print(f'Total columnas:    {len(COLUMN_NAMES)}')

## Sección 1: Inspección de Estructura e Integridad (FD002)

In [ ]:
train_file = DATA_DIR / 'train_FD002.txt'
df_train = pd.read_csv(train_file, sep=r'\s+', header=None, names=COLUMN_NAMES)

print(f'Total registros: {len(df_train):,}')
print(f'Motores: {df_train["unit_number"].nunique()}')
print(f'Valores nulos: {df_train.isnull().sum().sum()}')

In [ ]:
df_train.head()

In [ ]:
df_train.columns

## Sección 2: Clusterización de Regímenes Operativos (K-Means)

### Contexto y Fundamentación Física
A diferencia del subconjunto FD001 (que opera en una única condición estacionaria a nivel del mar), el dataset **FD002** simula una envolvente de vuelo realista con **6 condiciones operativas distintas**. Estas condiciones están gobernadas por tres variables de control ambiental y de comando:

* **`setting_1` (Altitud):** Altitud de vuelo, con rangos típicos de 0 a 42,000 pies (0 a 42 kft).
* **`setting_2` (Velocidad / Número Mach):** Velocidad relativa del aire, variando de 0.00 a 0.84 Mach.
* **`setting_3` (Ángulo de comando del acelerador / Throttle Resolver Angle - TRA):** Comandos de potencia discretos (ej. 60° y 100°).

En lugar de una variación continua e indeterminada, la turbina opera en **6 puntos de equilibrio cuasi-estacionarios** (combinaciones discretas de Altitud, Mach y TRA).

### Propósito del Análisis
Dado que cada régimen impone niveles de presión, densidad y temperatura base radicalmente distintos en el motor, el primer paso metodológico obligatorio es **identificar y etiquetar automáticamente a qué régimen pertenece cada ciclo de operación**.

Para ello, se estandarizan las tres variables operativas con `StandardScaler` (evitando que la escala numérica de la altitud en miles domine sobre el número Mach) y se aplica el algoritmo no supervisado **K-Means con $k=6$**.

In [ ]:
# Inicializar y ajustar el StandardScaler
scaler = StandardScaler()
settings_scaled = scaler.fit_transform(df_train[SETTING_COLS])

# Entrenar K-Means con k=6 para los regímenes del FD002
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_train['regime'] = kmeans.fit_predict(settings_scaled)

# 1. Preparación de datos
df_sample = df_train.sample(n=min(4000, len(df_train)), random_state=42).copy()
df_sample['regime_label'] = df_sample['regime'].apply(lambda x: f'Régimen {x}')

centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
df_centroids = pd.DataFrame(centroids_original, columns=['setting_1', 'setting_2', 'setting_3'])

# 2. Construcción de la figura
fig = go.Figure()

vibrant_colors = [
    '#E74C3C', 
    '#3498DB', 
    '#2ECC71', 
    '#9B59B6', 
    '#F39C12',
    '#1ABC9C'  
]

for i, regime_name in enumerate(sorted(df_sample['regime_label'].unique())):
    df_regime = df_sample[df_sample['regime_label'] == regime_name]
    
    fig.add_trace(go.Scatter3d(
        x=df_regime['setting_1'],
        y=df_regime['setting_2'],
        z=df_regime['setting_3'],
        mode='markers',
        marker=dict(
            size=5,  # <-- Puntos más grandes (Original era 3.5)
            color=vibrant_colors[i % len(vibrant_colors)],
            opacity=0.6,
            symbol='circle',
            line=dict(width=0) 
        ),
        name=regime_name
    ))

# 3. Centroides limpios y elegantes (sin mostrar en leyenda)
fig.add_trace(go.Scatter3d(
    x=df_centroids['setting_1'],
    y=df_centroids['setting_2'],
    z=df_centroids['setting_3'],
    mode='text+markers',
    text=[f'<b>C{i}</b>' for i in range(len(df_centroids))],
    textposition='top center',
    textfont=dict(size=13, color='black', family='Arial'),
    marker=dict(
        size=11,
        color=vibrant_colors[:len(df_centroids)],
        symbol='circle',
        line=dict(color='white', width=2) # Borde blanco limpio
    ),
    showlegend=False # Esto elimina la entrada de "Centroides" de la leyenda
))

fig.update_layout(
    width=1100,  
    height=700,  
    title=dict(
        text='<b>Identificación de Regímenes Operativos (C-MAPSS FD002)</b>',
        x=0.5,
        font=dict(size=22, family='Arial', color='#2c3e50')
    ),
    scene=dict(
        xaxis=dict(
            title='<b>Setting 1</b>', 
            backgroundcolor='white', 
            gridcolor='#e5e5e5', 
            showbackground=True,
            zerolinecolor='gray'
        ),
        yaxis=dict(
            title='<b>Setting 2</b>', 
            backgroundcolor='white', 
            gridcolor='#e5e5e5', 
            showbackground=True,
            zerolinecolor='gray'
        ),
        zaxis=dict(
            title='<b>Setting 3</b>', 
            backgroundcolor='white', 
            gridcolor='#e5e5e5', 
            showbackground=True,
            zerolinecolor='gray'
        ),
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=1.1) 
        )
    ),
    paper_bgcolor='white',
    legend=dict(
        title='<b>Leyenda</b>',
        yanchor='top', y=0.95,
        xanchor='left', x=1.02,
        font=dict(size=13),
        itemsizing='constant' 
    ),
    margin=dict(r=150, b=40, l=40, t=90)
)

try:
    import kaleido  # noqa: F401  -> exportador estático de Plotly
    fig.write_image(str(FIGURES_DIR / 'regimenes_clusters_3d_plotly.png'), scale=3)
    print('  Guardado: notebooks/figures/FD002/regimenes_clusters_3d_plotly.png')
except ImportError:
    print('[aviso] kaleido no instalado: se omite el PNG estático del 3D interactivo.')
    print('        (para habilitarlo: uv add kaleido)')
fig.show()

In [ ]:
# --- 1. PREPARACIÓN, ESCALADO Y K-MEANS ---
if 'SETTING_COLS' not in locals():
    SETTING_COLS = ['setting_1', 'setting_2', 'setting_3']

# Inicializar y ajustar el StandardScaler
scaler = StandardScaler()
settings_scaled = scaler.fit_transform(df_train[SETTING_COLS])

# Entrenar K-Means con k=6 para los regímenes del FD002
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_train['regime'] = kmeans.fit_predict(settings_scaled)
# ----------------------------------------------------

# --- 2. CONFIGURACIÓN Y VISUALIZACIÓN ---
fig = plt.figure(figsize=(12, 7))
ax = fig.add_subplot(111, projection='3d')

# Tomamos una muestra para no saturar la memoria visual
df_sample = df_train.sample(n=min(5000, len(df_train)), random_state=42)

# Paleta de colores profesionales y distinguibles para los 6 regímenes
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6', '#F39C12', '#1ABC9C']

# Graficar cada régimen por separado para que aparezca limpio en la leyenda
for regime_id in sorted(df_sample['regime'].unique()):
    df_regime = df_sample[df_sample['regime'] == regime_id]
    
    ax.scatter(
        df_regime['setting_1'], 
        df_regime['setting_2'], 
        df_regime['setting_3'], 
        c=colors[regime_id % len(colors)], 
        alpha=0.5, 
        s=50,
        label=f'Régimen {regime_id}'
    )

# Estética general de los ejes y el título
ax.set_title('Identificación de Regímenes Operativos (C-MAPSS FD002)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Setting 1', labelpad=10, fontweight='bold')
ax.set_ylabel('Setting 2', labelpad=10, fontweight='bold')
ax.set_zlabel('Setting 3', labelpad=10, fontweight='bold')

# Fondo blanco limpio (estilo reporte científico / tesis)
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Leyenda limpia al costado sin referencias a centroides
ax.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, fontsize=11)

plt.tight_layout()

# Guardar figura en alta resolución para la tesis
guardar_fig('regimenes_clusters_3d')
plt.show()

## Sección 3: Cálculo de Varianza Intra-Régimen

In [ ]:
# Calcular varianza global y varianza por régimen
variance_global = df_train[SENSOR_COLS].var()
variance_intra = df_train.groupby('regime')[SENSOR_COLS].var().mean()

comparison = pd.DataFrame({'Global': variance_global, 'Intra-Regimen': variance_intra})
comparison['Reduction'] = (1 - comparison['Intra-Regimen'] / comparison['Global']) * 100

print('Sensores con alta reducción de varianza (candidatos a limpieza intra-régimen):')
print(comparison.sort_values('Reduction', ascending=False).head(10))

### Interpretación de la Varianza: Ruido Operativo vs. Señal Real de Degradación

La tabla anterior compara la **Varianza Global** (calculada sobre todo el dataset sin distinguir condiciones de vuelo) frente a la **Varianza Intra-Régimen Media** (calculada agrupando las lecturas por cada uno de los 6 clústeres operativos y promediando su dispersión interna). La reducción porcentual de varianza se define como:

$$\text{Reducción (\%)} = \left(1 - \frac{\text{Varianza Intra-Régimen}}{\text{Varianza Global}}\right) \times 100$$

Este análisis arroja un hallazgo fundamental para el preprocesamiento de FD002: **permite clasificar los sensores en dos grupos con tratamientos metodológicos opuestos**, resolviendo el dilema de qué variables deben eliminarse y cuáles deben normalizarse.

### Hallazgos Clave de la Varianza Intra-Régimen

---

#### 1. Sensores Invariantes Intra-Régimen (Reducción = 100%, Varianza Intra $\approx 0$) $\rightarrow$ **Candidatos a Remoción Definitiva (`sensors.remove`)**

* **Sensores identificados:** `sensor_1`, `sensor_5`, `sensor_18`, `sensor_19` (junto con `sensor_16`, `sensor_6` y `sensor_10`, cuya variabilidad interna es prácticamente nula).
* **¿Qué significa físicamente?:** Cuando el motor opera dentro de un régimen específico, estos sensores **no varían en absoluto**; su lectura es una línea completamente plana a lo largo de toda la vida útil de la turbina.
* **¿Por qué exhibían varianza global?:** Su variabilidad a nivel global es un **falso positivo**. El sensor salta de un valor constante a otro únicamente cuando la aeronave cambia de altitud o velocidad (condiciones de entrada $T_2$, $P_2$, etc.), pero dentro de cualquier vuelo se mantiene estático.
* **Decisión metodológica para la tesis:** **Deben ser descartados.** Dado que no sufren cambios ante el desgaste del compresor HPC, no aportan ninguna información predictiva para estimar el RUL. Mantenerlos introduciría ruido escalonado espurio que perjudicaría a los algoritmos de Machine Learning. Estos sensores formarán la lista definitiva de `sensors.remove` en `configs/config_FD002.yaml`.

---

#### 2. Sensores Dinámicos Sensibles al Régimen (Reducción > 99.9%, pero Varianza Intra $> 0$) $\rightarrow$ **Candidatos a Normalización por Régimen**

* **Sensores identificados:** `sensor_2`, `sensor_3`, `sensor_4`, `sensor_7`, `sensor_8`, `sensor_11`, `sensor_12`, `sensor_13`, `sensor_15`, `sensor_17`, `sensor_20`, `sensor_21`.
* **¿Qué significa físicamente?:** Más del 99.9% de la fluctuación numérica que presentan en los datos crudos se debe exclusivamente al cambio entre condiciones operativas (por ejemplo, el `sensor_8` tiene una varianza global gigantesca de 21,085 por los cambios de altitud, pero su varianza intra-régimen es de apenas 0.04).
* **¿Contienen señal de degradación?:** **Sí.** A diferencia del Grupo 1, su varianza intra-régimen **no es cero**. Esa pequeña dispersión residual representa la termodinámica interna del desgaste progresivo del compresor.
* **Decisión metodológica para la tesis:** **No deben eliminarse.** Lo que se requiere es **desacoplar el régimen de vuelo** mediante técnicas de estandarización por clúster (Regime Normalization), eliminando los saltos operativos para dejar al descubierto la tendencia limpia de degradación.

---

### Resumen Metodológico 
En el dataset FD001 (un solo régimen), los sensores constantes se identificaban fácilmente con una simple varianza global igual a cero. En **FD002**, el análisis de varianza intra-régimen demuestra que los sensores **1, 5, 16, 18 y 19 son el equivalente exacto a los sensores constantes de FD001**, mientras que los sensores termodinámicos clave deben someterse a una transformación por régimen antes de alimentar cualquier modelo de pronóstico de RUL.

## Sección 4: Análisis de Degradación por Régimen

### Selección de Sensores Clave para la Inspección de Degradación

Para ilustrar de forma tangible el fenómeno de enmascaramiento operativo y validar la efectividad de la normalización por régimen, seleccionamos cuatro variables termodinámicas centrales del núcleo del motor:

$$\text{Sensores Clave} = [\text{sensor\_2}, \text{sensor\_3}, \text{sensor\_4}, \text{sensor\_11}]$$

Esta selección se fundamenta en dos pilares:

1. **Física del Modo de Falla (Compresor de Alta Presión - HPC):**
   * **`sensor_2` ($T_{24}$ - Temperatura total a la salida del LPC):** Refleja la redistribución de carga térmica en la etapa de compresión previa al núcleo.
   * **`sensor_3` ($T_{30}$ - Temperatura total a la salida del HPC):** Medición directa del gas comprimido al salir del HPC hacia la cámara de combustión. Al desgastarse el compresor por erosión o aumento de holgura en los álabes (*blade tip clearance*), la eficiencia adiabática cae y la temperatura de salida aumenta para un mismo trabajo de compresión.
   * **`sensor_4` ($T_{50}$ - Temperatura total a la salida de la LPT):** Para compensar la ineficiencia del compresor y mantener el empuje comandado, el sistema de control inyecta más combustible, elevando la temperatura de los gases que ingresan y salen de las turbinas.
   * **`sensor_11` ($P_{s30}$ - Presión estática a la salida del HPC):** Indicador crítico de la relación de compresión y del flujo másico del núcleo.

2. **Consenso en la Literatura Internacional:**
   Múltiples estudios de referencia en pronóstico de turbofanes (Saxena et al., 2008; Babu et al., 2016; Zheng et al., 2017; Li et al., 2018) han validado que estas cuatro mediciones son las que exhiben mayor consistencia y monotonicidad respecto al avance del deterioro en turbomaquinaria de dos ejes.

In [ ]:
key_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_11']

In [ ]:
for s in key_sensors:
    plt.figure(figsize=(12, 5))
    
    # Filtramos el motor 1
    df_m1 = df_train[df_train['unit_number'] == 1].sort_values('time_cycles')
    
    # Usamos scatterplot con una línea tenue que una los puntos cronológicamente
    sns.lineplot(data=df_m1, x='time_cycles', y=s, color='gray', alpha=0.3, linewidth=1, sort=False)
    
    scatter = sns.scatterplot(
        data=df_m1, 
        x='time_cycles', 
        y=s, 
        hue='regime', 
        palette='viridis', 
        s=25, 
        legend='full'
    )
    
    plt.title(f'Evolución Cronológica de {s} en Motor 1 (Coloreado por Régimen)', fontsize=14, fontweight='bold')
    plt.xlabel('Ciclos de Tiempo', fontweight='bold')
    plt.ylabel(s, fontweight='bold')
    
    # Ajustar la leyenda para que no estorbe
    plt.legend(title='Régimen', bbox_to_anchor=(1.02, 1), loc='upper left')
    
    plt.tight_layout()
    guardar_fig(f'degradacion_cronologica_{s}_regimen')
    plt.show()

### Diagnóstico Visual: Fenómeno de Enmascaramiento Operativo

Las gráficas anteriores trazan la trayectoria cronológica completa del **Motor 1** desde su primer vuelo (ciclo 1) hasta su falla final, con una línea tenue que conecta las lecturas sucesivas y puntos coloreados según el régimen de vuelo asignado por K-Means.

Al analizar estas trayectorias crudas, se evidencian dos fenómenos fundamentales:

1. **Dominancia del Ruido de Conmutación Operativa:**
   La señal experimenta oscilaciones verticales abruptas en forma de zig-zag continuo. Por ejemplo, en el `sensor_2`, la temperatura salta de ~550 °R a ~640 °R de un ciclo a otro. Estos saltos bruscos coinciden con exactitud matemática con los cambios de color (cambios de régimen). La amplitud del salto operativo (~90 °R) es hasta **un orden de magnitud superior** a la variación total producida por el desgaste (~10-15 °R).

2. **Tendencia de Degradación Subyacente pero Oculta:**
   A pesar de las violentas fluctuaciones entre regímenes, si se aíslan visualmente los puntos de un solo color (por ejemplo, siguiendo únicamente los puntos amarillos o verdes), se aprecia una suave deriva ascendente conforme el motor acumula ciclos. Sin embargo, en el conjunto global, esta señal de deterioro queda completamente **enmascarada** bajo la envolvente de vuelo.

### Fundamentación Metodológica: ¿Por qué es Indispensable la Normalización por Régimen?

#### 1. El Problema de la Ambigüedad Operativa (Falsa Salud vs. Falsa Falla)
Si se alimentara un modelo de Machine Learning (como XGBoost, Random Forest, Redes Neuronales o Memorias Asociativas) directamente con las mediciones crudas de FD002, el algoritmo fracasaría al intentar correlacionar el valor numérico con el RUL debido a la **ambigüedad contextual**:

$$\text{Lectura: } \text{sensor\_2} = 642^\circ\text{R}$$

* Si la aeronave vuela en el **Régimen 1** (baja altitud y alta velocidad, exigencia severa), $642^\circ\text{R}$ es el valor nominal de diseño de un **motor nuevo y saludable** ($\text{RUL} > 120$ ciclos).
* Si la aeronave vuela en el **Régimen 4** (gran altitud y crucero suave), la temperatura nominal debería rondar los $535^\circ\text{R}$. Por lo tanto, una lectura de $642^\circ\text{R}$ en esa condición representaría un sobrecalentamiento crítico de un **motor a punto de colapsar** ($\text{RUL} \approx 0$ ciclos).

Un modelo no normalizado no puede distinguir si un valor elevado proviene de una condición de vuelo exigente o de un motor degradado.

---

#### 2. La Solución Matemática: Estandarización por Régimen (Z-Score por Clúster)
Para desacoplar el efecto ambiental y aislar el desgaste interno, se aplica una transformación de estandarización condicional por régimen:

$$z_{i, t} = \frac{x_{i, t} - \mu_{k, i}}{\sigma_{k, i}} \quad \forall \quad t \in \text{Régimen } k$$

donde:
* $x_{i, t}$ es la lectura cruda del sensor $i$ en el ciclo $t$.
* $\mu_{k, i}$ es la media del sensor $i$ calculada exclusivamente sobre los datos del régimen operativo $k \in \{0, 1, \dots, 5\}$.
* $\sigma_{k, i}$ es la desviación estándar intra-régimen del sensor $i$ en dicho régimen.

---

#### 3. Beneficios Técnicos 
1. **Supresión del Sesgo Operativo:** Alinea todos los regímenes en una línea base neutral con media común igual a cero ($\mu = 0$).
2. **Emergencia de la Tendencia Monotónica:** Remueve los escalones artificiales del vuelo y expone la curva acumulativa del deterioro del compresor HPC, permitiendo que los modelos estimen el RUL con alta fidelidad y estabilidad numérica.

In [ ]:
# 1. Aplicar la normalización por régimen (Estandarización) para los sensores clave si no se ha hecho
key_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_11']

for s in key_sensors:
    mean_regime = df_train.groupby('regime')[s].transform('mean')
    std_regime = df_train.groupby('regime')[s].transform('std').replace(0, 1)
    df_train[f'{s}_norm'] = (df_train[s] - mean_regime) / std_regime

# 2. Paleta de colores vibrantes idéntica a tus gráficos anteriores
vibrant_colors = [
    '#E74C3C', # Rojo vivo
    '#3498DB', # Azul claro / brillante
    '#2ECC71', # Verde esmeralda
    '#9B59B6', # Morado / Púrpura
    '#F39C12', # Naranja intenso
    '#1ABC9C'  # Turquesa
]

# 3. Generar la gráfica con el mismo formato estético para el Motor 1
for s in key_sensors:
    plt.figure(figsize=(12, 6))
    
    # Filtramos el motor 1 y ordenamos cronológicamente
    df_m1 = df_train[df_train['unit_number'] == 1].sort_values('time_cycles').copy()
    norm_col = f'{s}_norm'
    
    # Línea de fondo tenue que conecta la evolución cronológica
    plt.plot(
        df_m1['time_cycles'], 
        df_m1[norm_col], 
        color='gray', 
        alpha=0.3, 
        linewidth=1, 
        zorder=1
    )
    
    # Puntos de dispersión coloreados por su régimen original para conservar la estética visual
    for i, regime_id in enumerate(sorted(df_m1['regime'].unique())):
        df_regime = df_m1[df_m1['regime'] == regime_id]
        
        plt.scatter(
            df_regime['time_cycles'], 
            df_regime[norm_col], 
            color=vibrant_colors[regime_id % len(vibrant_colors)], 
            s=30, 
            alpha=0.8, 
            edgecolors='white', 
            linewidths=0.5,
            label=f'Régimen {regime_id}',
            zorder=2
        )
    
    # Línea de referencia horizontal en 0 (la media esperada tras la estandarización)
    plt.axhline(0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    
    # Estética y formato profesional para la tesis
    plt.title(f'Evolución Estandarizada de {s} (Motor 1 - Sin Ruido Operativo)', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Ciclos de Tiempo', fontsize=12, fontweight='bold')
    plt.ylabel(f'{s} (Estandarizado)', fontsize=12, fontweight='bold')
    
    # Fondo blanco limpio estilo tesis
    plt.gca().set_facecolor('white')
    plt.grid(True, linestyle=':', alpha=0.5, color='gray')
    
    # Leyenda lateral limpia
    plt.legend(title='Regímen', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, fontsize=11)
    
    plt.tight_layout()
    guardar_fig(f'degradacion_estetica_normalizada_{s}')
    plt.show()

### Conclusiones y Validación: Señal Cruda vs. Señal Normalizada

La comparación directa entre la trayectoria cruda (Sección 4.1) y la trayectoria normalizada (Sección 4.2) valida empíricamente la hipótesis metodológica de este EDA:

1. **Eliminación Total del Ruido Escalonado:**
   Al contrastar ambas representaciones, se evidencia que los saltos masivos inter-régimen (que oscilaban erráticamente entre 530 y 640 en el `sensor_2`) han sido completamente neutralizados. Tras la estandarización condicional, los puntos de los 6 regímenes colapsan en una única envolvente centrada alrededor de cero ($z = 0$) durante toda la etapa inicial de operación.

2. **Emergencia de la Curva de Deterioro Bifásica:**
   En las series normalizadas se revela con total nitidez la evolución de la salud del motor:
   * **Fase Estable / Saludable (Ciclos 1 a ~100):** La señal oscila de forma estacionaria con media cero dentro de un ancho de banda normal ($[-1, +1]$), indicando que el motor opera bajo desgaste nominal sin daño perceptible.
   * **Fase de Degradación Acelerada (Ciclos ~100 hasta la falla):** La trayectoria quiebra la horizontal y experimenta un incremento monotónico y continuo, superando $+2$ y $+3$ desviaciones estándar al aproximarse al ciclo final. Este ascenso sostenido es la **firma termodinámica real del deterioro del compresor de alta presión (HPC)**.

---

### Implicaciones Directas para la Arquitectura del Proyecto

* **Validación del Piecewise Linear RUL ($RUL_{max} = 125$):**
  La existencia de esta fase inicial plana seguida de un declive progresivo confirma que la función objetivo de Piecewise Linear RUL utilizada en FD001 sigue siendo completamente válida para FD002.
* **Diseño del Pipeline de Datos (`src/data/pipeline.py`):**
  Queda demostrado que para FD002 y FD004 es **estrictamente obligatorio** aplicar la etapa de clusterización por K-Means y la normalización Z-score por régimen antes de calcular ventanas temporales (*sliding windows*) o entrenar los modelos de regresión.
* **Configuración del Experimento (`configs/config_FD002.yaml`):**
  Los sensores con varianza intra-régimen nula (`sensor_1`, `sensor_5`, `sensor_16`, `sensor_18`, `sensor_19`) deben registrarse formalmente en la clave `sensors.remove` para ser excluidos de todo el flujo de modelado.